# MOMENT embeddings — DIMER live tutorial

This notebook extracts pooled representations from the pinned **AutonLab/MOMENT-1-base** encoder. It uses the reproducible repository runtime, verifies a deterministic bundled sample (or accepts BYOD), runs CPU inference, and exports embeddings plus provenance.

**Important:** upstream `MOMENT.embed` has no per-point missingness mask. Missing-value fractions are recorded, but pre-filled positions are visible to the encoder. The tutorial therefore uses a clean sample by default.


## 1. Bootstrap the repository and locked runtime


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kurtvalcorza/moment-pipeline.git"
REPO_NAME = "moment-pipeline"

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL], check=True)
    os.chdir(REPO_NAME)
    ROOT = Path.cwd()
    subprocess.run(
        ["uv", "pip", "install", "--system", "-r", "requirements.lock.txt"],
        check=True,
    )
    subprocess.run(
        ["uv", "pip", "install", "--system", "--no-deps", "-e", "."],
        check=True,
    )
else:
    print(f"Repository checkout detected: {ROOT}")


## 2. Load the deterministic sample or BYOD


In [ ]:
import hashlib
import io
import json

import pandas as pd

USE_BYOD = False

if USE_BYOD:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("BYOD upload is available when this notebook runs in Colab.") from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV.")
    name, payload = next(iter(uploaded.items()))
    frame = pd.read_csv(io.BytesIO(payload))
    print(f"Loaded BYOD: {name}")
else:
    sample_root = ROOT / "examples" / "sample-data"
    subprocess.run([sys.executable, str(sample_root / "generate_samples.py")], check=True)
    sample_path = sample_root / "moment_clean.csv"
    manifest = {}
    for line in (sample_root / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
        digest, filename = line.split("  ", 1)
        manifest[filename] = digest
    observed = hashlib.sha256(sample_path.read_bytes()).hexdigest()
    assert observed == manifest[sample_path.name]
    frame = pd.read_csv(sample_path)
    print(f"Loaded verified synthetic sample: {sample_path}")

frame["timestamp"] = pd.to_datetime(frame["timestamp"])
print(frame.head())
print(f"rows={len(frame)}, channels={frame['channel'].nunique()}")


## 3. Validate and canonicalize


In [ ]:
from moment_pipeline import build_provenance, embed, load_moment, to_windows

windows = to_windows(frame)
print("window tensor:", windows.x_enc.shape)
print("channels:", windows.channels)
print("source missing fraction:", windows.masked_point_fraction)


## 4. Resolve the pinned checkpoint and extract embeddings


In [ ]:
model = load_moment(task="embedding", device="cpu")
result = embed(windows, model, warmup=False)
provenance = build_provenance(model, windows, result)

print("embedding shape:", result.embeddings.shape)
print("reduction:", result.reduction)
print("channel policy:", result.channel_policy)
print("revision:", model.identity.revision)


## 5. Inspect and export


In [ ]:
output_dir = ROOT / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

embedding_frame = result.to_frame()
embedding_frame.to_csv(output_dir / "moment_embeddings.csv", index=False)
(output_dir / "moment_embeddings_provenance.json").write_text(
    json.dumps(provenance, indent=2, default=str),
    encoding="utf-8",
)

print(embedding_frame.iloc[:, :8])
print("embedding L2 norm:", float((result.embeddings[0] ** 2).sum() ** 0.5))
print("exports:", sorted(path.name for path in output_dir.glob("moment_embeddings*")))


## Interpretation

The output is one vector per canonical window after MOMENT's `reduction="mean"` policy, which averages channels inside upstream before patch pooling. It is a pretrained representation, **not a classifier prediction** and not evidence that missing values were ignored.
